# Challenge 6b: Classifying Gas Turbine Operational State from Temperature Sensors

**Author:** Dr Rob Lyon

**Version:** 1.0

## Code & License
Copyright &copy; 2026 Robert Lyon. All Rights Reserved.

Permission is granted to use, copy, and modify this material for
personal educational purposes only. Redistribution, resale, or use
in commercial or institutional settings requires prior written
permission from the author. This material is provided for educational
purposes as part of a structured course. The author accepts no
liability for incorrect results or decisions arising from use of this
material. All datasets used in this challenge are synthetic and do
not represent proprietary or confidential experimental data.

## Table of Contents

1. [Introduction](#1-introduction)
2. [Useful Links](#2-useful-links)
3. [Libraries and Environment Setup](#3-libraries-and-environment-setup)
   - [3.1 Working Environment](#31-working-environment)
   - [3.2 Libraries Used](#32-libraries-used)
   - [3.3 Data](#33-data)
   - [3.4 Imports](#34-imports)
4. [The Data](#4-the-data)
   - [4.1 Loading the Data](#41-loading-the-data)
   - [4.2 Understanding the Features](#42-understanding-the-features)
   - [4.3 Exploring the Data](#43-exploring-the-data)
   - [4.4 Data Quality: Missing Values and Outliers](#44-data-quality-missing-values-and-outliers)
   - [4.5 Preprocessing](#45-preprocessing)
5. [The Challenge](#5-the-challenge)
   - [5.1 Training the Model](#51-training-the-model)
   - [5.2 Evaluating the Model](#52-evaluating-the-model)
   - [5.3 Interpretation](#53-interpretation)
   - [5.4 Reflection Questions](#54-reflection-questions)
6. [Solution](#6-solution)
7. [References](#7-references)

---
## 1. Introduction

A heavy-duty industrial gas turbine generating 200 megawatts runs at
compressor discharge temperatures exceeding 400 degrees Celsius and
turbine inlet temperatures above 1300 degrees. The blades rotating at
3000 rpm in that environment are made from single-crystal nickel
superalloys cooled internally by air bled from the compressor. They
operate within 50 to 100 degrees of their metallurgical limit at all
times. When blade metal temperatures exceed that limit for even a few
minutes, creep deformation begins. Continued operation leads to blade
fracture, which at turbine speeds is a catastrophic mechanical failure.

Operators prevent this by monitoring a distributed array of temperature
sensors continuously and classifying the turbine's thermal state in
real time. Four states matter operationally:

- **Normal**: all temperatures within the design envelope. The unit
  can continue operating at current load.
- **Warning**: one or more sensors elevated beyond normal operating
  range, indicating increased load, inlet fouling, or early-stage hot
  section degradation. Operator attention is required; load reduction
  may be warranted.
- **Critical**: temperature thresholds have been breached. The control
  system is issuing alarms. Intervention is required; the unit may need
  to be tripped if temperatures do not recover.
- **Fault**: sensor or mechanical failure producing erratic or
  physically implausible readings. The most operationally ambiguous
  state: a fault reading could mean a failed sensor (false alarm) or
  a real runaway event being misreported by a dying thermocouple.

Operators managing multiple units simultaneously cannot watch every sensor trend
continuously. An automated classifier that flags Warning and Critical
states reliably, and correctly identifies Fault as distinct from
Critical, directly reduces the risk of undetected blade failure.

The dataset you will work with, **TurbineMonitor**, contains 10,000
observations from a simulated heavy-duty industrial gas turbine
instrumented with seven temperature sensors. It has two important
data quality issues that must be handled before any model can be fit:

- **Missing values**: the blade metal temperature (`tbmt_c`) and
  inter-stage cooling supply (`ics_c`) sensors have missing rates of
  12.5% and 8.8% respectively. Both are in a restricted instrumentation
  bundle where cable damage is common. Bearing temperature (`brg_c`)
  has a 3% missing rate from vibration-induced dropouts. These are not
  random failures: the missingness in `ics_c` is partly conditional on
  `tbmt_c` being missing (same cable run). You cannot simply drop rows
  with missing values without losing a large fraction of the dataset.
- **Sensor outliers**: approximately 2.5% of rows contain spike values
  from electromagnetic interference or transient electrical faults.
  These produce readings far outside the physically plausible range for
  that sensor. They are not related to the true operational state and
  will mislead any model that treats them as genuine signal.

You will apply `GaussianNB` from scikit-learn. As covered in Notebook_6,
Gaussian Naive Bayes models each feature as a Gaussian distribution
conditioned on the class, then classifies using Bayes' theorem. It
assumes features are conditionally independent given the class. That
assumption is violated here: exhaust gas temperature and blade metal
temperature are strongly correlated because both reflect the same
underlying hot section thermal state. This violation is a deliberate
part of the challenge. You will see its consequences in the model's
performance and interpret what it means in this domain.

### Learning Objectives

- Apply `GaussianNB` to a four-class safety-critical classification
  problem with real-world data quality issues
- Detect, characterise, and handle missing values using median imputation
  via `SimpleImputer`, applied correctly without data leakage
- Detect and cap sensor outliers using IQR-based bounds fitted only on
  training data
- Evaluate a multiclass classifier on an imbalanced dataset (5:20:50:25)
  using per-class precision, recall, and F1 score
- Interpret the per-class Gaussian parameters learned by `GaussianNB`
  (class means and variances) in the context of gas turbine physics
- Reason about the consequences of the feature independence assumption
  being violated in a correlated sensor dataset

---
## 2. Useful Links

| Resource | URL |
|---|---|
| `GaussianNB` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html |
| `SimpleImputer` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html |
| Missing value strategies (scikit-learn user guide) | https://scikit-learn.org/stable/modules/impute.html |
| `classification_report` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html |
| `ConfusionMatrixDisplay` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html |
| Gas turbine condition monitoring overview (NASA) | https://ntrs.nasa.gov/citations/20000023918 |
| IEC 62740: Root cause analysis standard (safety systems) | https://www.iec.ch/homepage |

---
## 3. Libraries and Environment Setup

### 3.1 Working Environment

This notebook requires Python 3.9 or later. All libraries are part of
the standard scientific Python stack. No specialist engineering or
signal processing library is required.

### 3.2 Libraries Used

| Library | Purpose |
|---|---|
| `numpy` | Numerical operations and outlier detection |
| `pandas` | Loading, inspecting, and cleaning the dataset |
| `matplotlib` | Plotting |
| `seaborn` | Distribution plots and heatmaps |
| `sklearn.model_selection` | Stratified train/test splitting |
| `sklearn.impute` | `SimpleImputer` for missing value handling |
| `sklearn.naive_bayes` | `GaussianNB` classifier |
| `sklearn.metrics` | Confusion matrix, classification report, accuracy |

### 3.3 Data

| Property | Value |
|---|---|
| Filename | `temp_monitoring.csv` |
| Location | `data/temp_monitoring.csv` (relative to this notebook) |
| Total rows | 10,000 |
| Features | 7 temperature sensor columns (all in degrees Celsius; see Section 4.2) |
| Target column | `op_state` |
| Class 0 | normal: ~4,885 samples (~48.9%) |
| Class 1 | warning: ~2,611 samples (~26.1%) |
| Class 2 | critical: ~1,987 samples (~19.9%) |
| Class 3 | fault: ~517 samples (~5.2%) |

The dataset is moderately imbalanced. The fault class has only ~5% of
samples, reflecting how rarely a real turbine enters a true fault state.
This rarity makes it the hardest class to classify correctly, and the
one where misclassification has the most severe consequences.

### 3.4 Imports

In [ ]:
# Listing 1.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

CLASS_NAMES = ['normal', 'warning', 'critical', 'fault']
FEATURE_COLS = ['egt_c', 'cit_c', 'tbmt_c', 'brg_c', 'ics_c', 'lube_c', 'amb_c']

print('Libraries loaded successfully.')

---
## 4. The Data

### 4.1 Loading the Data

In [ ]:
# Listing 2.
df = pd.read_csv('data/temp_monitoring.csv')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print()
df.head(10)

### 4.2 Understanding the Features

All seven features are temperature measurements from different locations
in and around the gas turbine. They are measured in degrees Celsius
unless stated otherwise. The sensors vary in reliability, physical
location, and what aspect of turbine health they reflect.

| Feature | Units | Normal range | Description |
|---|---|---|---|
| `egt_c` | deg C | 480-560 | Exhaust Gas Temperature. Measured at the turbine exit. The primary indicator of turbine load and hot section health. The most reliable sensor; almost never missing. |
| `cit_c` | deg C | 5-40 | Compressor Inlet Temperature. Ambient air temperature entering the compressor. Varies with weather; largely independent of turbine internal thermal state. Reliable. |
| `tbmt_c` | deg C | 750-920 | Turbine Blade Metal Temperature. The most critical measurement operationally. Requires optical pyrometry or embedded thermocouples in rotating blades, making it the most difficult sensor to maintain. Missing rate ~12.5%. |
| `brg_c` | deg C | 60-95 | Bearing Temperature. Elevated values indicate lubrication breakdown or mechanical wear. Subject to vibration-induced dropout (~3% missing). |
| `ics_c` | deg C | 290-370 | Inter-stage Cooling Supply Temperature. Temperature of compressor bleed air used to cool internal turbine components. Missing at ~8.8%, partially conditional on `tbmt_c` missingness (same cable bundle). |
| `lube_c` | deg C | 45-75 | Lube Oil Temperature. Correlated with bearing temperature; both rise together under lubrication stress. Rarely missing. |
| `amb_c` | deg C | -5 to 45 | Ambient Temperature. Dry-bulb temperature of the surrounding environment. Used for inlet condition correction. Largely uncorrelated with internal turbine temperatures. Very reliable. |

### 4.3 Exploring the Data

In [ ]:
# Listing 3.
print('Class distribution:')
counts = df['op_state'].value_counts().sort_index()
for label, count in counts.items():
    print(f'  Class {label} ({CLASS_NAMES[label]:10s}): {count:5d}  ({100*count/len(df):.1f}%)')

print()
print('Mean feature values by class (complete cases):')
print(df.groupby('op_state')[FEATURE_COLS].mean().round(1).to_string())

**Questions to consider:**

- The Normal, Warning, and Critical classes show a clear progression in
  `egt_c`, `tbmt_c`, and `brg_c`. Does this match the physical description
  in Section 4.2?
- The Fault class (class 3) has lower mean `egt_c` and `tbmt_c` than the
  Critical class. Why? Recall from the introduction that the Fault state
  includes both runaway mechanical failures (high readings) and sensor
  failures (near-zero readings). What does averaging these two sub-types
  produce?
- `cit_c` and `amb_c` means are similar across all four classes. Does
  this make physical sense, given what these sensors measure?

In [ ]:
# Listing 4.
# Feature distributions by class
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
colours = {0: 'steelblue', 1: 'goldenrod', 2: 'firebrick', 3: 'purple'}

for ax, feat in zip(axes, FEATURE_COLS):
    for cls in [0, 1, 2, 3]:
        subset = df.loc[df['op_state'] == cls, feat].dropna()
        subset.plot(kind='kde', ax=ax, label=CLASS_NAMES[cls],
                    color=colours[cls], linewidth=1.8)
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel('')
    ax.legend(fontsize=8)

# Hide the unused 8th subplot
axes[-1].set_visible(False)

plt.suptitle('Feature Distributions by Operational State (TurbineMonitor)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

**Questions to consider:**

- The `tbmt_c` distribution for the Fault class is clearly bimodal: one
  peak at very high temperatures (mechanical failure) and one near zero
  (sensor failure). Gaussian Naive Bayes models each class distribution
  as a single Gaussian. What will happen when it tries to fit a Gaussian
  to this bimodal distribution? Will the resulting model be a good
  representation of the Fault class?
- `egt_c`, `tbmt_c`, and `ics_c` all show the same ordering of class
  peaks (Normal < Warning < Critical). What does this tell you about
  the information content of these three features relative to each other?
- `cit_c` and `amb_c` distributions are nearly identical across classes.
  Do you expect GaussianNB to use these features heavily, or will they
  contribute near-zero discriminating power?

In [ ]:
# Listing 5.
# Correlation matrix (using complete cases)
corr = df[FEATURE_COLS].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, linewidths=0.5, ax=ax, square=True
)
ax.set_title('Feature Correlation Matrix (TurbineMonitor)', fontsize=12)
plt.tight_layout()
plt.show()

**Questions to consider:**

- `egt_c`, `tbmt_c`, and `ics_c` are strongly correlated (around 0.8-0.87).
  Explain this physically: what do all three sensors have in common in
  terms of what they measure about the turbine's thermal state?
- `brg_c` and `lube_c` are strongly correlated (around 0.78). What
  physical process links bearing temperature and lube oil temperature?
- `cit_c` and `amb_c` are correlated with each other (~0.79) but
  largely uncorrelated with the hot-section sensors. Why?
- Gaussian Naive Bayes assumes conditional independence between features
  given the class. The correlation matrix shows this assumption is clearly
  violated for several feature pairs. What consequence do you expect this
  to have for the model's probability estimates, even if classification
  accuracy remains reasonable?

### 4.4 Data Quality: Missing Values and Outliers

Before preprocessing, you must understand the extent and pattern of the
data quality issues. This section provides the diagnostic tools. You
will handle both issues in Section 4.5.

In [ ]:
# Listing 6.
# Missing value audit
print('Missing value counts per feature:')
missing = df[FEATURE_COLS].isnull().sum()
missing_pct = 100 * missing / len(df)
miss_df = pd.DataFrame({'count': missing, 'percent': missing_pct.round(2)})
print(miss_df[miss_df['count'] > 0].to_string())

print(f'\nTotal rows with at least one missing value: '
      f'{df[FEATURE_COLS].isnull().any(axis=1).sum()} '
      f'({100*df[FEATURE_COLS].isnull().any(axis=1).mean():.1f}%)')

print('\nConditional missingness: rows where both tbmt_c and ics_c are missing:')
both_missing = df['tbmt_c'].isnull() & df['ics_c'].isnull()
print(f'  {both_missing.sum()} rows ({100*both_missing.mean():.1f}%)')
print(f'  Of rows where tbmt_c is missing, fraction also missing ics_c: '
      f'{(df.loc[df["tbmt_c"].isnull(), "ics_c"].isnull().mean()):.2f}')

**Questions to consider:**

- Which two features have the highest missing rates? Does this match the
  physical explanation in Section 4.2 (restricted instrumentation bundle)?
- The conditional missingness analysis shows that when `tbmt_c` is missing,
  a substantial fraction of `ics_c` values are also missing. What does this
  tell you about the independence of the missing value mechanism? Is this
  data Missing Completely At Random (MCAR)?
- If you simply dropped all rows containing at least one missing value,
  what fraction of the dataset would you lose? Would the remaining dataset
  be a representative sample of the original?

In [ ]:
# Listing 7.
# Outlier detection using IQR bounds
print('Outlier detection (values outside 1.5 * IQR from Q1/Q3):')
print()
for feat in FEATURE_COLS:
    col = df[feat].dropna()
    q1, q3 = col.quantile(0.25), col.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_out = ((col < lower) | (col > upper)).sum()
    if n_out > 0:
        print(f'  {feat:10s}: {n_out:4d} outliers  '
              f'(bounds: [{lower:.1f}, {upper:.1f}],  '
              f'actual range: [{col.min():.1f}, {col.max():.1f}])')

**Questions to consider:**

- The outlier values are far outside the physically plausible operating
  range for each sensor. For example, an `egt_c` value above 900 C or
  below 150 C cannot reflect genuine gas turbine operation (the turbine
  would have tripped automatically). This tells you these are measurement
  artifacts rather than genuine signal.
- If you fit a Gaussian to a feature that contains spike outliers, what
  effect will the outliers have on the estimated mean and especially the
  estimated variance? Will the Gaussian accurately represent the bulk of
  the data?
- Consider the difference between a Fault class reading of, say, 1350 C
  on `tbmt_c` (mechanical runaway) and a sensor spike outlier of 1380 C
  on a Normal class sample. Both are extreme values, but one is genuine
  class signal and the other is a measurement artifact. How would you
  decide which is which in a real deployment?

### 4.5 Preprocessing

Two data quality issues require handling before the data can be passed
to `GaussianNB`.

**Outlier treatment:** Cap outliers at IQR-based bounds. This preserves
the row rather than dropping it, and replaces the physically implausible
spike value with the nearest plausible boundary value. The alternative,
dropping outlier rows, discards genuine class signal along with the
artifact. Fitting the bounds on training data only and applying them to
the test set prevents leakage: if you compute IQR bounds on the whole
dataset, test-set statistics influence the training-set preprocessing.

**Missing value imputation:** Replace missing values with the median of
the training set for that feature using `SimpleImputer(strategy='median')`.
Median imputation is robust to the outliers that remain in the raw data
after capping (the median is not pulled by extreme values the way the
mean is). As with outlier bounds, the imputer must be fit on training
data only and then applied to both training and test sets.

The order matters: cap outliers first, then impute. If you impute first,
extreme spike values will distort the median estimate for features with
high missing rates. If you cap first, the imputed median reflects the
cleaned distribution.

`GaussianNB` does not require feature scaling: it estimates each
feature's mean and variance directly from the data, and the classification
decision depends on the ratio of likelihoods rather than on absolute
magnitudes. Scaling the features would not change the result.

In [ ]:
# Listing 8.
# TODO: Implement preprocessing.
#
# Step 1: Split the data.
#   Separate features (X) from the target ('op_state') (y).
#   Use an 80/20 train/test split, random_state=42, stratify by y.
#   Print class counts in y_train and y_test to confirm stratification.
#
# Step 2: Compute IQR-based outlier bounds from X_train only.
#   For each feature column:
#     - Compute Q1 and Q3 on the training data (ignoring NaNs)
#     - Compute IQR = Q3 - Q1
#     - Set lower bound = Q1 - 1.5 * IQR
#     - Set upper bound = Q3 + 1.5 * IQR
#   Store these bounds (you will need them to clip X_test).
#   Apply np.clip to X_train using the bounds.
#   Apply the same bounds (fitted on X_train) to X_test.
#   Print how many values were clipped in training and test.
#
# Step 3: Apply median imputation.
#   Instantiate SimpleImputer(strategy='median').
#   Fit on X_train_clipped, then transform both X_train_clipped and
#   X_test_clipped.
#   Verify there are no remaining NaNs in either set after imputation.
#
# Note: do not scale the features. Explain in a comment why scaling
# is unnecessary for GaussianNB.

# YOUR CODE HERE

---
## 5. The Challenge

Your task is to train a `GaussianNB` classifier on the preprocessed
TurbineMonitor data and evaluate how well it distinguishes the four
operational states.

`GaussianNB` fits a Gaussian distribution to each feature for each
class, then uses Bayes' theorem to compute the posterior probability
of each class given an observation. The class with the highest
posterior probability is the prediction. Because it models each
feature independently per class, it is fast to train, interpretable,
and works well even with moderate sample sizes for each class. Its
weakness is the independence assumption: when features are strongly
correlated, as several are here, the model double-counts evidence.
The probabilities it outputs will be poorly calibrated even when the
most-likely class is correctly identified.

### 5.1 Training the Model

In [ ]:
# Listing 9.
# TODO: Train the GaussianNB model.
#
# 1. Instantiate GaussianNB().
#    There are no critical hyperparameters to tune for a first fit.
#    You may optionally set var_smoothing (the default 1e-9 is fine).
#
# 2. Fit the model on your imputed, clipped training data.
#
# 3. After fitting, print:
#    - clf.classes_            : the class labels the model knows about
#    - clf.theta_.shape        : shape of the learned mean matrix
#                                (should be n_classes x n_features = 4 x 7)
#    - clf.var_.shape          : same shape, for the learned variances
#    Inspect clf.theta_: each row is the per-class mean vector for each
#    feature. Do the means for egt_c increase from Normal to Critical?

# YOUR CODE HERE

### 5.2 Evaluating the Model

With four classes and a 5:20:50:25 imbalance, overall accuracy conflates
performance across very differently sized groups. The Fault class has only
~5% of samples: a model that never predicts Fault would still achieve ~95%
accuracy on Fault rows by ignoring them while getting everything else right.
Per-class recall is the metric that exposes this.

In a safety-critical context, the costs of error are asymmetric:

- Missing a Critical event (predicting Warning when the true state is
  Critical) risks undetected blade temperature exceedance.
- Misclassifying a Fault as Critical is an over-alarm: it triggers
  unnecessary intervention but does not endanger the turbine.
- Misclassifying a Fault as Normal is the most dangerous error: the
  operator takes no action during a potential runaway event.

The confusion matrix makes all of these error types visible at once.

In [ ]:
# Listing 10.
# TODO: Evaluate the model.
#
# 1. Generate predictions on the imputed, clipped test set.
#
# 2. Print overall accuracy.
#
# 3. Print the full classification report with target_names=CLASS_NAMES.
#    Pay particular attention to Fault (class 3):
#    - What is its recall? Are missed Fault events being predicted as
#      Critical, Warning, or Normal?
#    - What is its precision? Is the model generating false Fault alarms?
#
# 4. Plot the confusion matrix using ConfusionMatrixDisplay.from_predictions().
#    Use display_labels=CLASS_NAMES and a colour map that makes small
#    counts visible (e.g. cmap='Blues').
#    Which off-diagonal cell has the largest count?

# YOUR CODE HERE

### 5.3 Interpretation

`GaussianNB` learns two parameters per feature per class: the mean
(`clf.theta_`) and variance (`clf.var_`). These are stored as matrices
of shape (n_classes, n_features). After imputation and outlier capping,
these parameters represent the Gaussian fitted to each feature
distribution within each class.

Inspecting them lets you check whether the model has learned physically
sensible distributions, and whether the overlap between class distributions
explains the errors you observed in the confusion matrix.

In [ ]:
# Listing 11.
# TODO: Inspect the learned Gaussian parameters.
#
# 1. Create a DataFrame from clf.theta_ (the class means).
#    Use CLASS_NAMES as the row index and FEATURE_COLS as the columns.
#    Print it with two decimal places.
#    Do the egt_c means increase monotonically from Normal to Critical?
#    What about the Fault class mean?
#
# 2. Create a DataFrame from clf.var_ (the class variances) in the same way.
#    Print it. The Fault class will have very high variance for most features.
#    Explain why: think about what the Fault class distribution looks like
#    from Section 4.3.
#
# 3. Compute the per-class standard deviations (sqrt of variances) for egt_c.
#    For each class pair (e.g. Warning vs Critical), compute the overlap
#    region qualitatively: if the means are close relative to the standard
#    deviations, the Gaussians overlap substantially.
#    Which class pair has the most overlapping egt_c distributions?

# YOUR CODE HERE

In [ ]:
# Listing 12.
# TODO: Visualise the learned Gaussians for one feature.
#
# Choose egt_c (exhaust gas temperature) as the feature to visualise.
#
# For each class (0-3), plot the Gaussian PDF with the learned mean and
# standard deviation (sqrt of variance) over the range 0 to 900.
# Use the class colours from the exploration section if possible.
# Add a legend and label the axes clearly.
# Title the plot: 'GaussianNB learned distributions: egt_c'.
#
# Do the learned Gaussians look like reasonable fits to the actual
# distributions you saw in Section 4.3?
# For the Fault class, does a single Gaussian adequately represent
# the bimodal distribution?

# YOUR CODE HERE

### 5.4 Reflection Questions

1. The Fault class has a bimodal distribution in most features (one peak
   from mechanical runaway, one from sensor failure near zero). `GaussianNB`
   fits a single Gaussian to each class. What effect does this have on the
   learned variance for Fault, and on the classifier's ability to correctly
   identify Fault observations near both peaks?

2. The conditional independence assumption is clearly violated: `egt_c`,
   `tbmt_c`, and `ics_c` are correlated at around 0.8. If you removed two
   of the three and kept only one, do you expect classification accuracy
   to change significantly? Why or why not?

3. You used median imputation for the missing values. The missing rate for
   `tbmt_c` is ~12.5% and the missingness in `ics_c` is partly conditional
   on `tbmt_c` being missing. Does median imputation respect this conditional
   structure, or does it impute both features independently? What would a
   more sophisticated imputation strategy look like?

4. The outlier capping strategy preserves rows by replacing spike values
   with the IQR boundary. For a Fault class sample that has a genuine
   runaway reading of 1380 C on `tbmt_c`, capping clips this to the upper
   IQR boundary computed from the training data (which is dominated by
   Normal and Warning samples). What does this do to the Fault class signal?
   Is there a better strategy for handling Fault-class extremes?

5. In the safety context described in the introduction, which type of error
   is more costly: classifying a Critical state as Warning, or classifying
   a Warning state as Critical? How would you modify the classifier to
   reflect this asymmetry? (Hint: consider `priors` in `GaussianNB`, or
   adjusting the decision threshold on predicted probabilities.)

---
## 6. Solution

**Read this section only after attempting the challenge yourself.**
Every step is explained with the reasoning behind each decision.
The cells run from top to bottom without depending on variables
from Section 5.

### Step 1: Train/Test Split

We split before any preprocessing to prevent leakage. All parameters
(IQR bounds, imputation medians) will be computed from training data
only and then applied identically to the test set. Stratification
preserves the 5% fault class in both splits; without it, a random split
could place very few fault samples in the test set.

In [ ]:
# Listing 13.
X = df[FEATURE_COLS].values
y = df['op_state'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape},  Test set: {X_test.shape}')
print()
print('Class counts in training set:')
for cls in range(4):
    n = np.sum(y_train == cls)
    print(f'  {CLASS_NAMES[cls]:10s}: {n:5d}  ({100*n/len(y_train):.1f}%)')
print('Class counts in test set:')
for cls in range(4):
    n = np.sum(y_test == cls)
    print(f'  {CLASS_NAMES[cls]:10s}: {n:5d}  ({100*n/len(y_test):.1f}%)')

### Step 2: Outlier Capping

IQR bounds are computed from `X_train` ignoring NaNs (using `np.nanpercentile`).
The bounds are stored and then applied to both training and test arrays.
We clip before imputing because we want the median imputed values to reflect
the cleaned, non-spiked distribution. Imputing first would pull the median
of `egt_c` toward the centre of the full range including spikes, slightly
biasing every imputed value upward or downward.

Note that for the Fault class, genuine extreme readings (e.g. a true runaway
reading of 1350 C) will also be clipped. This is a trade-off: we lose some
Fault-class signal in order to clean up the artifact spikes. In a production
system you would apply class-specific bounds or use a more sophisticated
anomaly detector to distinguish genuine extreme values from sensor faults.
For this exercise, global IQR bounds computed on the training mix are the
appropriate starting point.

In [ ]:
# Listing 14.
# Compute IQR bounds from training data
lower_bounds = np.zeros(len(FEATURE_COLS))
upper_bounds = np.zeros(len(FEATURE_COLS))

for i in range(len(FEATURE_COLS)):
    col = X_train[:, i]
    q1 = np.nanpercentile(col, 25)
    q3 = np.nanpercentile(col, 75)
    iqr = q3 - q1
    lower_bounds[i] = q1 - 1.5 * iqr
    upper_bounds[i] = q3 + 1.5 * iqr

# Apply bounds: clip without touching NaNs
def clip_with_nan(X, lower, upper):
    """Clip values to [lower, upper] per column, preserving NaN entries."""
    X_clipped = X.copy().astype(float)
    for i in range(X.shape[1]):
        mask = ~np.isnan(X_clipped[:, i])
        X_clipped[mask, i] = np.clip(X_clipped[mask, i], lower[i], upper[i])
    return X_clipped

X_train_clip = clip_with_nan(X_train, lower_bounds, upper_bounds)
X_test_clip  = clip_with_nan(X_test,  lower_bounds, upper_bounds)

# Count clipped values (non-NaN values that were changed)
n_clipped_train = int(np.nansum(
    (X_train.astype(float) < lower_bounds) | (X_train.astype(float) > upper_bounds)
))
n_clipped_test = int(np.nansum(
    (X_test.astype(float) < lower_bounds) | (X_test.astype(float) > upper_bounds)
))
print(f'Values clipped in training set: {n_clipped_train}')
print(f'Values clipped in test set:     {n_clipped_test}')

### Step 3: Median Imputation

The imputer is fit on `X_train_clip` and applied to both sets.
Median is preferred over mean here because the outlier capping is
IQR-based (robust), but residual extreme values from the Fault class
may still sit near the clip boundary and skew a mean estimate.
The median is unaffected by these remaining extremes.

In [ ]:
# Listing 15.
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_clip)
X_test_imp  = imputer.transform(X_test_clip)

print(f'NaNs remaining in training set: {np.isnan(X_train_imp).sum()}')
print(f'NaNs remaining in test set:     {np.isnan(X_test_imp).sum()}')
print()
print('Imputation medians per feature (from training data):')
for feat, med in zip(FEATURE_COLS, imputer.statistics_):
    print(f'  {feat:10s}: {med:.2f}')

### Step 4: Training

`GaussianNB` requires no hyperparameter tuning for a first fit. It
estimates the per-class feature means and variances directly from the
training data in a single pass: this is one of its major practical
advantages over gradient-based methods.

In [ ]:
# Listing 16.
clf = GaussianNB()
clf.fit(X_train_imp, y_train)

print(f'Classes: {clf.classes_}')
print(f'Learned mean matrix shape (theta_):  {clf.theta_.shape}  (n_classes x n_features)')
print()
print('Per-class means (clf.theta_):')
theta_df = pd.DataFrame(clf.theta_, index=CLASS_NAMES, columns=FEATURE_COLS)
print(theta_df.round(2).to_string())

The mean matrix should show a clear monotonic progression from Normal to
Critical in `egt_c`, `tbmt_c`, and `ics_c`. The Fault class mean sits
between Warning and Critical for most features: this is the average of
the high-temperature mechanical fault sub-population and the near-zero
sensor failure sub-population, pulled toward the middle. The high variance
the model assigns to Fault (below) is the consequence of this averaging.

### Step 5: Evaluation

In [ ]:
# Listing 17.
y_pred = clf.predict(X_test_imp)

acc = accuracy_score(y_test, y_pred)
print(f'Overall accuracy: {acc:.4f}\n')
print('Classification report:')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

Normal and Warning are classified well. Critical is also strong. Fault
is the challenging class: its precision and recall are lower than the
others, driven by two factors.

First, the bimodal Fault distribution: `GaussianNB` fits a single
Gaussian per feature per class. The Fault Gaussian has a very high
variance because it is trying to span both the high-temperature
mechanical failure sub-population and the near-zero sensor failure
sub-population. This wide, flat Gaussian overlaps with every other
class, making the Fault likelihood ambiguous everywhere.

Second, the class prior: with only 5% of samples, the prior probability
of Fault is low. Even when the likelihood ratio slightly favours Fault,
the low prior can push the posterior toward a more common class. The
model is, in probabilistic terms, correctly reflecting the rarity of
genuine fault events. Whether this behaviour is acceptable in a
safety-critical system is a deployment decision, not a model quality
judgment.

In [ ]:
# Listing 18.
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=CLASS_NAMES,
    cmap='Blues',
    ax=ax
)
ax.set_title('Confusion Matrix: TurbineMonitor\n(GaussianNB, median imputation)', fontsize=10)
plt.tight_layout()
plt.show()

The confusion matrix shows that most errors fall along the adjacent-class
pairs (Normal/Warning, Warning/Critical). Cross-class errors (Normal
misclassified as Critical) are rare, reflecting the clear separation
between the extreme classes in most features. Fault errors scatter across
multiple classes because the wide Fault Gaussian overlaps with all of them.

### Step 6: Interpreting the Learned Parameters

In [ ]:
# Listing 19.
print('Per-class variances (clf.var_):')
var_df = pd.DataFrame(clf.var_, index=CLASS_NAMES, columns=FEATURE_COLS)
print(var_df.round(2).to_string())
print()
print('Per-class standard deviations:')
std_df = var_df.pow(0.5)
print(std_df.round(2).to_string())

The Fault class has by far the highest variance in every feature. This
is expected: a single Gaussian is being asked to represent a mixture of
near-zero sensor failure readings and high-temperature mechanical failure
readings. The resulting fitted Gaussian has a mean somewhere between
the two modes and a very large variance that covers both, and the
intervening region where neither mode actually sits. This is a direct
consequence of fitting a unimodal model to a bimodal class.

In [ ]:
# Listing 20.
# Visualise learned Gaussians for egt_c
from scipy.stats import norm as scipy_norm

x_range = np.linspace(0, 900, 500)
colours_plot = {0: 'steelblue', 1: 'goldenrod', 2: 'firebrick', 3: 'purple'}

fig, ax = plt.subplots(figsize=(10, 5))
egt_idx = FEATURE_COLS.index('egt_c')

for cls in range(4):
    mean = clf.theta_[cls, egt_idx]
    std  = np.sqrt(clf.var_[cls, egt_idx])
    pdf  = scipy_norm.pdf(x_range, mean, std)
    ax.plot(x_range, pdf, label=f'{CLASS_NAMES[cls]} (mu={mean:.0f}, sigma={std:.0f})',
            color=colours_plot[cls], linewidth=2)

ax.set_xlabel('egt_c (deg C)', fontsize=11)
ax.set_ylabel('Probability Density', fontsize=11)
ax.set_title('GaussianNB learned distributions: egt_c (Exhaust Gas Temperature)', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

The plot shows the four Gaussians the model uses for classification decisions
based on `egt_c` alone. Normal, Warning, and Critical form a clear sequence
with moderate overlap between adjacent classes. The Fault Gaussian is wide
and flat: its large variance means it assigns moderate likelihood to almost
the entire range. This is why Fault precision is lower than the other classes:
the model assigns Fault likelihood to observations that actually belong to
other classes, generating false positive Fault alarms, while also sometimes
preferring Warning or Critical over Fault for genuine fault observations
because their narrower Gaussians produce higher peak likelihoods near the
true Fault readings.

---
## 7. References

1. Zhang, H. (2004). The optimality of Naive Bayes. In *Proceedings of the
   17th International Florida Artificial Intelligence Research Society
   Conference (FLAIRS 2004)*. AAAI Press.
   Analysis of why Naive Bayes often performs well despite violating the
   independence assumption.

2. Domingos, P. and Pazzani, M. (1997). On the optimality of the simple
   Bayesian classifier under zero-one loss. *Machine Learning*, 29, 103-130.
   https://doi.org/10.1023/A:1007413511361
   Foundational paper on the conditions under which Naive Bayes remains
   competitive with correlated features.

3. Jaw, L.C. and Mattingly, J.D. (2009). *Aircraft Engine Controls:
   Design, System Analysis, and Health Monitoring*.
   AIAA Education Series. American Institute of Aeronautics and Astronautics.
   Reference for gas turbine control systems, sensor arrays, and operational
   state classification in real turbine monitoring environments.

4. Li, Y.G. (2010). Gas turbine performance and health status estimation
   using adaptive gas path analysis. *Journal of Engineering for Gas
   Turbines and Power*, 132(4), 041701.
   https://doi.org/10.1115/1.3159378
   Peer-reviewed example of machine learning applied to turbine health
   monitoring using temperature and pressure sensor data.

5. van Buuren, S. (2018). *Flexible Imputation of Missing Data* (2nd ed.).
   CRC Press. https://stefvanbuuren.name/fimd/
   Comprehensive reference on missing data mechanisms (MCAR, MAR, MNAR)
   and imputation strategies, including conditional missingness patterns.

6. Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python.
   *Journal of Machine Learning Research*, 12, 2825-2830.
   https://jmlr.org/papers/v12/pedregosa11a.html
   Cite this when reporting results obtained with scikit-learn.